In [1]:
from pathlib import Path
import pandas as pd
import math

In [2]:
BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name != "MMSR":
    BASE_DIR = BASE_DIR.parent

EVALUATION_PATH = BASE_DIR / "results" / "evaluation" / "evaluation_results.csv"
TFIDF_PATH = BASE_DIR / "results" / "tfidf" / "tfidf_results.csv"
TRANSFORMER_PATH = BASE_DIR / "results" / "transformer" / "transformer_results.csv"

In [3]:
evaluation_df = pd.read_csv(EVALUATION_PATH, keep_default_na=False)
tfidf_df = pd.read_csv(TFIDF_PATH)
transformer_df = pd.read_csv(TRANSFORMER_PATH)

print("Evaluation rows:", len(evaluation_df))
print("TF-IDF result rows:", len(tfidf_df))
print("Transformer result rows:", len(transformer_df))

Evaluation rows: 60
TF-IDF result rows: 30
Transformer result rows: 30


In [4]:
required_evaluation_columns = {
    "system", "query_id", "query", "rank", "source_row_id",
    "title", "similarity_score", "synopsis", "relevance"
}

required_result_columns = {
    "query_id", "query", "rank", "source_row_id", "title", "similarity_score"
}

print("Evaluation columns valid:", required_evaluation_columns.issubset(evaluation_df.columns))
print("TF-IDF columns valid:", required_result_columns.issubset(tfidf_df.columns))
print("Transformer columns valid:", required_result_columns.issubset(transformer_df.columns))
print("Evaluation rows valid:", len(evaluation_df) == 60)
print("TF-IDF rows valid:", len(tfidf_df) == 30)
print("Transformer rows valid:", len(transformer_df) == 30)

Evaluation columns valid: True
TF-IDF columns valid: True
Transformer columns valid: True
Evaluation rows valid: True
TF-IDF rows valid: True
Transformer rows valid: True


In [5]:
evaluation_df["relevance"] = pd.to_numeric(evaluation_df["relevance"], errors="raise").astype(int)
valid_relevance = evaluation_df["relevance"].isin([0, 1, 2]).all()
print("All relevance values valid:", valid_relevance)
print("Relevance distribution:")
display(evaluation_df["relevance"].value_counts().sort_index().rename_axis("relevance").reset_index(name="count"))

All relevance values valid: True
Relevance distribution:


,relevance,count
0,0,16
1,1,19
2,2,25


In [6]:
def dcg_at_5(relevances):
    return sum((2 ** relevance - 1) / math.log2(position + 1) for position, relevance in enumerate(relevances, start=1))

def ndcg_at_5(relevances):
    actual = dcg_at_5(relevances)
    ideal = dcg_at_5(sorted(relevances, reverse=True))
    return actual / ideal if ideal > 0 else 0.0

def precision_at_5(relevances):
    return sum(value > 0 for value in relevances) / 5

In [7]:
metric_rows = []

for system in ["TF-IDF", "Transformer"]:
    system_df = evaluation_df[evaluation_df["system"] == system]
    for query_id in sorted(system_df["query_id"].unique()):
        query_df = system_df[system_df["query_id"] == query_id].sort_values("rank")
        relevances = query_df["relevance"].tolist()
        metric_rows.append({
            "system": system,
            "query_id": int(query_id),
            "query": query_df.iloc[0]["query"],
            "top1_relevance": relevances[0],
            "precision_at_5": precision_at_5(relevances),
            "dcg_at_5": dcg_at_5(relevances),
            "ndcg_at_5": ndcg_at_5(relevances)
        })

query_metrics = pd.DataFrame(metric_rows)
display(query_metrics)

,system,query_id,query,top1_relevance,precision_at_5,dcg_at_5,ndcg_at_5
0,TF-IDF,1,കുടുംബ ബന്ധങ്ങളും കുടുംബ പ്രശ്നങ്ങളും,2,1.0,5.948459,0.957835
1,TF-IDF,2,പ്രണയവും പ്രണയ ബന്ധങ്ങളും,2,0.8,6.952588,0.904717
2,TF-IDF,3,കുറ്റകൃത്യങ്ങളും ത്രില്ലർ കഥയും,0,0.2,1.500000,0.500000
3,TF-IDF,4,സൗഹൃദവും സുഹൃത്തുക്കളുടെ ബന്ധവും,1,0.6,3.452588,0.640223
4,TF-IDF,5,ആക്ഷനും പ്രതികാരവും,0,0.6,2.517783,0.609495
5,TF-IDF,6,സാമൂഹിക പ്രശ്നങ്ങളും കുടുംബ നാടകവും,1,1.0,5.210319,0.838978
6,Transformer,1,കുടുംബ ബന്ധങ്ങളും കുടുംബ പ്രശ്നങ്ങളും,1,1.0,5.071672,0.816652
7,Transformer,2,പ്രണയവും പ്രണയ ബന്ധങ്ങളും,2,0.8,6.083518,0.891558
8,Transformer,3,കുറ്റകൃത്യങ്ങളും ത്രില്ലർ കഥയും,2,0.6,5.392789,1.000000
9,Transformer,4,സൗഹൃദവും സുഹൃത്തുക്കളുടെ ബന്ധവും,2,0.8,5.222165,0.896745


In [8]:
comparison = query_metrics.groupby("system", as_index=False).agg(
    queries_evaluated=("query_id", "nunique"),
    mean_top1_relevance=("top1_relevance", "mean"),
    mean_precision_at_5=("precision_at_5", "mean"),
    mean_dcg_at_5=("dcg_at_5", "mean"),
    mean_ndcg_at_5=("ndcg_at_5", "mean")
)

display(comparison.round(4))

,system,queries_evaluated,mean_top1_relevance,mean_precision_at_5,mean_dcg_at_5,mean_ndcg_at_5
0,TF-IDF,6,1.0,0.7000,4.2636,0.7419
1,Transformer,6,1.5,0.7667,5.2100,0.8721


In [9]:
overlap_rows = []

for query_id in sorted(tfidf_df["query_id"].unique()):
    tfidf_ids = set(tfidf_df[tfidf_df["query_id"] == query_id]["source_row_id"])
    transformer_ids = set(transformer_df[transformer_df["query_id"] == query_id]["source_row_id"])
    overlap_rows.append({
        "query_id": int(query_id),
        "query": tfidf_df[tfidf_df["query_id"] == query_id].iloc[0]["query"],
        "top5_overlap": len(tfidf_ids & transformer_ids)
    })

overlap_df = pd.DataFrame(overlap_rows)
display(overlap_df)
print("Mean Top-5 overlap:", round(overlap_df["top5_overlap"].mean(), 4))

,query_id,query,top5_overlap
0,1,കുടുംബ ബന്ധങ്ങളും കുടുംബ പ്രശ്നങ്ങളും,0
1,2,പ്രണയവും പ്രണയ ബന്ധങ്ങളും,0
2,3,കുറ്റകൃത്യങ്ങളും ത്രില്ലർ കഥയും,0
3,4,സൗഹൃദവും സുഹൃത്തുക്കളുടെ ബന്ധവും,0
4,5,ആക്ഷനും പ്രതികാരവും,1
5,6,സാമൂഹിക പ്രശ്നങ്ങളും കുടുംബ നാടകവും,0


Mean Top-5 overlap: 0.1667
